# Bronze Layer Monitor — `LEAGUE_RECORDS.BRONZE`
Health check for the **BRONZE schema** (raw ingestion layer). Covers tables, streams, stages, pipes, file formats, and ingestion state.

## How to Use
- Click **Run All** to execute every cell top-to-bottom. Or run individual cells to inspect specific objects.

In [ ]:
%%sql
USE DATABASE LEAGUE_RECORDS;

USE SCHEMA BRONZE;

---
## 1. Object Inventory
All objects registered in the BRONZE schema, grouped by type.

In [ ]:
%%sql
SELECT
    TABLE_TYPE AS OBJECT_TYPE,
    TABLE_NAME AS OBJECT_NAME,
    ROW_COUNT,
    ROUND(BYTES / 1024, 0)::INTEGER AS SIZE_KB,
    CREATED,
    LAST_ALTERED,
    COMMENT
FROM LEAGUE_RECORDS.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'BRONZE'
ORDER BY TABLE_TYPE, TABLE_NAME;

In [ ]:
%%sql
SHOW STAGES IN SCHEMA LEAGUE_RECORDS.BRONZE;

In [ ]:
%%sql
SHOW PIPES IN SCHEMA LEAGUE_RECORDS.BRONZE;

In [ ]:
%%sql
SHOW STREAMS IN SCHEMA LEAGUE_RECORDS.BRONZE;

In [ ]:
%%sql
SHOW FILE FORMATS IN SCHEMA LEAGUE_RECORDS.BRONZE;

---
## 2. Row Counts & Health
Row counts across all 5 bronze tables.

In [ ]:
SELECT 'MATCHES' AS TABLE_NAME, COUNT(*) AS ROW_COUNT 
FROM BRONZE.MATCHES
    UNION ALL
SELECT 'PLAYERS', COUNT(*) 
FROM BRONZE.PLAYERS
    UNION ALL
SELECT 'INTERVALS', COUNT(*) 
FROM BRONZE.INTERVALS
    UNION ALL
SELECT 'ITEMS_REF', COUNT(*) 
FROM BRONZE.ITEMS_REF
    UNION ALL
SELECT 'CHAMPIONS_REF', COUNT(*) 
FROM BRONZE.CHAMPIONS_REF;

---
## 3. Pipe Status & Ingestion History
Current pipe execution state and recent copy history for all bronze pipes.

In [ ]:
SELECT 
    PIPE_NAME,
    STATUS:executionState::VARCHAR AS EXECUTION_STATE,
    STATUS:pendingFileCount::INTEGER AS PENDING_FILE_COUNT,
    STATUS:lastIngestedFilePath::VARCHAR AS LAST_INGESTED_FILE_PATH,
    STATUS:lastIngestedTimestamp::TIMESTAMP_LTZ AS LAST_INGESTED_TIMESTAMP
FROM (
    SELECT 
        'MATCHES_PP' AS PIPE_NAME, 
        PARSE_JSON(SYSTEM$PIPE_STATUS('BRONZE.MATCHES_PP')) AS STATUS
    UNION ALL
    SELECT 'PLAYERS_PP', PARSE_JSON(SYSTEM$PIPE_STATUS('BRONZE.PLAYERS_PP'))
    UNION ALL
    SELECT 'INTERVALS_PP', PARSE_JSON(SYSTEM$PIPE_STATUS('BRONZE.INTERVALS_PP'))
    UNION ALL
    SELECT 'ITEMS_REF_PP', PARSE_JSON(SYSTEM$PIPE_STATUS('BRONZE.ITEMS_REF_PP'))
    UNION ALL
    SELECT 'CHAMPIONS_REF_PP', PARSE_JSON(SYSTEM$PIPE_STATUS('BRONZE.CHAMPIONS_REF_PP'))
)

In [ ]:
SELECT 
    TABLE_NAME,
    FILE_NAME,
    ROW_COUNT,
    ROW_PARSED,
    ERROR_COUNT,
    LAST_LOAD_TIME,
    STATUS
FROM (
    SELECT *
    FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
        TABLE_NAME => 'BRONZE.MATCHES',
        START_TIME => DATEADD(DAY, -14, CURRENT_TIMESTAMP())
    ))
        UNION ALL
    SELECT *
    FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
        TABLE_NAME => 'BRONZE.PLAYERS',
        START_TIME => DATEADD(DAY, -14, CURRENT_TIMESTAMP())
    ))
        UNION ALL
    SELECT *
    FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
        TABLE_NAME => 'BRONZE.INTERVALS',
        START_TIME => DATEADD(DAY, -14, CURRENT_TIMESTAMP())
    ))
        UNION ALL
    SELECT *
    FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
        TABLE_NAME => 'BRONZE.ITEMS_REF',
        START_TIME => DATEADD(DAY, -14, CURRENT_TIMESTAMP())
    ))
        UNION ALL
    SELECT *
    FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
        TABLE_NAME => 'BRONZE.CHAMPIONS_REF',
        START_TIME => DATEADD(DAY, -14, CURRENT_TIMESTAMP())
    ))
)
ORDER BY TABLE_NAME ASC, LAST_LOAD_TIME DESC
;

---
## 4. Stage File Inspection
Files currently present in each bronze stage.

In [ ]:
SELECT 
    'MATCHES_STG' AS STAGE, 
    RELATIVE_PATH, 
    ROUND(SIZE / 1024, 0)::INTEGER AS SIZE_KB, 
    LAST_MODIFIED
FROM DIRECTORY(@BRONZE.MATCHES_STG)
    UNION ALL
SELECT 'PLAYERS_STG', RELATIVE_PATH, ROUND(SIZE / 1024, 0)::INTEGER, LAST_MODIFIED
FROM DIRECTORY(@BRONZE.PLAYERS_STG)
    UNION ALL
SELECT 'INTERVALS_STG', RELATIVE_PATH, ROUND(SIZE / 1024, 0)::INTEGER, LAST_MODIFIED
FROM DIRECTORY(@BRONZE.INTERVALS_STG)
    UNION ALL
SELECT 'ITEMS_REF_STG', RELATIVE_PATH, ROUND(SIZE / 1024, 0)::INTEGER, LAST_MODIFIED
FROM DIRECTORY(@BRONZE.ITEMS_REF_STG)
    UNION ALL
SELECT 'CHAMPIONS_REF_STG', RELATIVE_PATH, ROUND(SIZE / 1024, 0)::INTEGER, LAST_MODIFIED
FROM DIRECTORY(@BRONZE.CHAMPIONS_REF_STG)

ORDER BY STAGE, RELATIVE_PATH;

---
## 5. Stream State
Check if any bronze streams have unconsumed data pending for silver.

In [ ]:
%%sql
SELECT 'MATCHES_STM' AS STREAM, SYSTEM$STREAM_HAS_DATA('BRONZE.MATCHES_STM') AS HAS_DATA
    UNION ALL
SELECT 'PLAYERS_STM', SYSTEM$STREAM_HAS_DATA('BRONZE.PLAYERS_STM')
    UNION ALL
SELECT 'INTERVALS_STM', SYSTEM$STREAM_HAS_DATA('BRONZE.INTERVALS_STM')
    UNION ALL
SELECT 'ITEMS_REF_STM', SYSTEM$STREAM_HAS_DATA('BRONZE.ITEMS_REF_STM')
    UNION ALL
SELECT 'CHAMPIONS_REF_STM', SYSTEM$STREAM_HAS_DATA('BRONZE.CHAMPIONS_REF_STM');

---
## 6. Data Previews
Quick samples to verify schema shape and data integrity.

In [ ]:
SELECT * 
FROM BRONZE.MATCHES
ORDER BY LDTS DESC
LIMIT 5;

In [ ]:
SELECT * 
FROM BRONZE.PLAYERS 
ORDER BY LDTS DESC
LIMIT 5;

In [ ]:
SELECT * 
FROM BRONZE.INTERVALS 
ORDER BY LDTS DESC
LIMIT 5;

In [ ]:
SELECT * 
FROM BRONZE.ITEMS_REF 
ORDER BY LDTS DESC
LIMIT 5;

In [ ]:
%%sql
SELECT * 
FROM BRONZE.CHAMPIONS_REF 
ORDER BY LDTS DESC
LIMIT 5;